# Dependencies

In [1]:
!pip install tensorflow opencv-python mediapipe scikit-learn numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.3 MB/s eta 0:00:00m eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 1.2 MB/s eta 0:00:00m eta 0:00:010:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 4.9 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 5.6 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 6.5 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 6.7 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 5.9 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 4.5 MB/s eta 0:00:00


In [1]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from tensorflow.keras.utils import to_categorical

2025-12-30 18:36:34.038461: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-30 18:36:34.101212: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-30 18:36:35.614623: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


# Hand Landmarker

In [2]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

BaseOptions = python.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.6,
    min_hand_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

hand_landmarker = HandLandmarker.create_from_options(options)

def detect_hands(frame, timestamp_ms):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    return hand_landmarker.detect_for_video(mp_image, timestamp_ms)



def draw_hand_landmarks(image, detection_result):
    if not detection_result.hand_landmarks:
        return image

    for hand_landmarks in detection_result.hand_landmarks:
        for lm in hand_landmarks:
            h, w, _ = image.shape
            cx, cy = int(lm.x * w), int(lm.y * h)
            cv2.circle(image, (cx, cy), 4, (0,255,0), -1)

    return image

def extract_hand_features(detection_result):
    if not detection_result.hand_landmarks:
        return np.zeros(21 * 3)

    hand = detection_result.hand_landmarks[0]
    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand])

    # Normalizar respecto a la muñeca
    coords -= coords[0]

    return coords.flatten()


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1767116196.147335  145457 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767116196.162766  145463 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


# Preparing Data

## Config

In [3]:
actions = np.array([
	'jump',
	'shoot',
	'none'
])

DATA_PATH = 'HandGestureData'
frame_interval = 1 / 30  # segundos entre frames (~30 FPS)
sequence_length = 15
no_sequences = 5

## Creating Folders

In [11]:
for action in actions:
    for seq in range(no_sequences):
        os.makedirs(os.path.join(DATA_PATH, action, str(seq)), exist_ok=True)

## Collecting Data

In [12]:
import cv2
import numpy as np
import os
import time


# Abrir cámara
cap = cv2.VideoCapture(0)
cv2.namedWindow('Grabación', cv2.WINDOW_NORMAL)

print("INSTRUCCIONES: Presiona 's' para iniciar cada repetición, 'q' para salir.")

stop_recording = False  # Variable global de control

for action in actions:
    if stop_recording:
        break
    print(f"\nPróximo gesto: {action}")

    for seq in range(no_sequences):
        if stop_recording:
            break

        started = False

        # Esperar a que pulses 's'
        while not started:
            ret, frame = cap.read()
            if not ret:
                continue

            cv2.putText(frame, f'Próximo gesto: {action} — Presiona S para iniciar',
                        (10,50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
            cv2.imshow('Grabación', frame)

            key = cv2.waitKey(10) & 0xFF
            if key == ord('s'):
                started = True
                print(f"Iniciando grabación de {action}, secuencia {seq+1}")
            elif key == ord('q'):
                stop_recording = True
                break

        if stop_recording:
            break

        # Grabación por intervalos de tiempo
        frames_captured = 0
        last_time = time.time()
        while frames_captured < sequence_length:
            if stop_recording:
                break

            current_time = time.time()
            if current_time - last_time >= frame_interval:
                last_time = current_time

                ret, frame = cap.read()
                if not ret:
                    continue

                timestamp = int(current_time * 1000)
                result = detect_hands(frame, timestamp)
                frame_display = draw_hand_landmarks(frame, result)

                cv2.putText(frame_display, f'{action} — secuencia {seq+1} frame {frames_captured+1}/{sequence_length}',
                            (10,50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
                cv2.imshow('Grabación', frame_display)

                features = extract_hand_features(result)
                np.save(os.path.join(DATA_PATH, action, str(seq), f'{frames_captured}.npy'), features)

                frames_captured += 1

            # Revisar tecla q cada frame
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                stop_recording = True
                break

        if stop_recording:
            break

        print(f"Secuencia completada: {seq+1}/{no_sequences}")
        print("Pulsa 's' para iniciar la siguiente repetición...")

cap.release()
cv2.destroyAllWindows()
print("Grabación terminada")


INSTRUCCIONES: Presiona 's' para iniciar cada repetición, 'q' para salir.

Próximo gesto: jump
Iniciando grabación de jump, secuencia 1
Secuencia completada: 1/5
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 2
Secuencia completada: 2/5
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 3
Secuencia completada: 3/5
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 4
Secuencia completada: 4/5
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 5
Secuencia completada: 5/5
Pulsa 's' para iniciar la siguiente repetición...

Próximo gesto: shoot
Iniciando grabación de shoot, secuencia 1
Secuencia completada: 1/5
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de shoot, secuencia 2
Secuencia completada: 2/5
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de shoot, secuencia 3
Secuencia completada

## Data Augmentation

In [13]:
def augment_sequence(seq):
    seq = np.array(seq)

    # Ruido espacial
    seq += np.random.normal(0, 0.01, seq.shape)

    # Escalado
    scale = np.random.uniform(0.9, 1.1)
    seq *= scale

    # Frame dropout
    if np.random.rand() < 0.3:
        idx = np.random.randint(0, len(seq))
        seq[idx] = 0

    return seq

# Creating Dataset

In [14]:
# 1️⃣ Construir y augmentar dataset (lo que ya tienes)
label_map = {label:i for i,label in enumerate(actions)}

sequences, labels = [], []

for action in actions:
    for seq in os.listdir(os.path.join(DATA_PATH, action)):
        window = []
        for frame in range(sequence_length):
            window.append(
                np.load(os.path.join(DATA_PATH, action, seq, f'{frame}.npy'))
            )

        sequences.append(window)
        labels.append(label_map[action])

        # Augmented copy
        sequences.append(augment_sequence(window))
        labels.append(label_map[action])

# 2️⃣ Guardar dataset en un solo archivo .npz
np.savez_compressed(
    'hand_gesture_dataset.npz',
    X=np.array(sequences),
    y=np.array(labels)
)
print("Dataset guardado en 'hand_gesture_dataset.npz' con X.shape =", np.array(sequences).shape)


Dataset guardado en 'hand_gesture_dataset.npz' con X.shape = (30, 15, 63)


# Creating Splits

In [15]:
# 3️⃣ Cargar dataset desde archivo
data = np.load('hand_gesture_dataset.npz')
X = data['X']
y_labels = data['y']

# Convertir a one-hot
y = to_categorical(y_labels, num_classes=len(actions))

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y_labels
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (25, 15, 63) X_test: (5, 15, 63)


# Creating Model

In [16]:
MODEL_EXPORT_NAME = 'hand_gesture_model.h5'

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GRU, TimeDistributed

model = Sequential([
    TimeDistributed(Dense(64, activation='relu'),
                    input_shape=(sequence_length, X.shape[2])),
    Dropout(0.3),

    GRU(64),
    Dense(32, activation='relu'),
    Dense(len(actions), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2025-12-30 18:41:17.625273: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 15, 64)         │         4,096 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 15, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,235 (122.01 KB)

 Trainable params: 31,235 (122.01 KB)

 Non-trainable params: 0 (0.00 B)

# Training

In [18]:
model.fit(
    X_train, y_train,
    epochs=150,
    validation_split=0.2,
    batch_size=16
)

model.save(MODEL_EXPORT_NAME)

Epoch 1/150
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 562ms/step - accuracy: 0.6000 - loss: 1.0666 - val_accuracy: 0.8000 - val_loss: 1.0346
Epoch 2/150
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.6500 - loss: 1.0295 - val_accuracy: 0.2000 - val_loss: 1.0372
Epoch 3/150
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.4000 - loss: 1.0174 - val_accuracy: 0.2000 - val_loss: 1.0357
Epoch 4/150
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.4000 - loss: 0.9961 - val_accuracy: 0.2000 - val_loss: 1.0137
Epoch 5/150
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.4000 - loss: 0.9734 - val_accuracy: 0.2000 - val_loss: 0.9951
Epoch 6/150
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.4000 - loss: 0.9553 - val_accuracy: 0.2000 - val_loss: 0.9639
Epoch 7/150
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.4500 - loss: 0.9316 - val_accuracy: 0.8000 - val_loss: 0.9194
Epoch 8/150
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.6000 - loss: 0.9010 - val_accuracy: 0.8000 - val_lo

# Metrics

In [19]:

from tensorflow.keras.models import load_model

model = load_model(MODEL_EXPORT_NAME)

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print("Accuracy:", accuracy_score(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 352ms/step
Accuracy: 1.0
[[2 0 0]
 [0 2 0]
 [0 0 1]]


# Testing

In [26]:

from tensorflow.keras.models import load_model

model = load_model(MODEL_EXPORT_NAME)

import time

sequence = []
pred_buffer = []
threshold = 0.75

cap = cv2.VideoCapture(0)

last_timestamp = int(time.time() * 1000)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # timestamp monotónico en ms
    timestamp_ms = int(time.time() * 1000)
    if timestamp_ms <= last_timestamp:
        timestamp_ms = last_timestamp + 1
    last_timestamp = timestamp_ms

    result = detect_hands(frame, timestamp_ms)
    frame = draw_hand_landmarks(frame, result)

    features = extract_hand_features(result)
    sequence.append(features)
    sequence = sequence[-sequence_length:]

    if len(sequence) == sequence_length:
        res = model.predict(np.expand_dims(sequence, axis=0))[0]
        pred_buffer.append(res)

        avg = np.mean(pred_buffer[-5:], axis=0)
        idx = np.argmax(avg)

        if avg[idx] > threshold and actions[idx] != 'none':
            cv2.putText(frame, actions[idx],
                        (50,50), cv2.FONT_HERSHEY_SIMPLEX,
                        1.5, (0,255,0), 3)
            # 🔥 aquí llamas a tu lógica del juego

    cv2.imshow('Feed', frame)
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 346ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━